In [1]:
import ee
import pandas as pd
from datetime import datetime, timedelta

# ============================================================
# 1. INITIALIZE GEE
# ============================================================

ee.Initialize(project="gis-project-work")

print("✅ Google Earth Engine initialized")


# ============================================================
# 2. JOSHIMATH LOCATION
# ============================================================

joshimath = ee.Geometry.Point([
    79.56436,
    30.55543
])

# 10 km study area
roi = joshimath.buffer(10000)

print("✅ Joshimath ROI ready")


# ============================================================
# 3. LIVE RAINFALL — GPM IMERG V07
# ============================================================

gpm = (
    ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
    .filterBounds(roi)
    .sort("system:time_start", False)
)

latest_rain_image = ee.Image(
    gpm.first()
)

rain_date = ee.Date(
    latest_rain_image.get("system:time_start")
)

rainfall = (
    latest_rain_image
    .select("precipitation")
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10000,
        bestEffort=True
    )
)

rainfall_value = rainfall.get(
    "precipitation"
).getInfo()

rain_date_value = rain_date.format(
    "YYYY-MM-dd HH:mm"
).getInfo()

print("\n🌧️ LIVE RAINFALL")
print("Date:", rain_date_value)
print("Rainfall:", rainfall_value)


# ============================================================
# 4. LIVE SOIL MOISTURE — SMAP
# ============================================================

smap = (
    ee.ImageCollection(
        "NASA/SMAP/SPL3SMP_E/005"
    )
    .filterBounds(roi)
    .sort("system:time_start", False)
)

latest_sm_image = ee.Image(
    smap.first()
)

sm_date = ee.Date(
    latest_sm_image.get("system:time_start")
)

soil = (
    latest_sm_image
    .select("soil_moisture_am")
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10000,
        bestEffort=True
    )
)

soil_value = soil.get(
    "soil_moisture_am"
).getInfo()

sm_date_value = sm_date.format(
    "YYYY-MM-dd"
).getInfo()

print("\n💧 LIVE SOIL MOISTURE")
print("Date:", sm_date_value)
print("Soil moisture:", soil_value)


# ============================================================
# 5. DEM — SRTM
# ============================================================

dem = ee.Image(
    "USGS/SRTMGL1_003"
)

dem_result = (
    dem
    .select("elevation")
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=30,
        bestEffort=True
    )
)

elevation = dem_result.get(
    "elevation"
).getInfo()

print("\n⛰️ DEM")
print("Mean elevation:", elevation, "m")


# ============================================================
# 6. SLOPE
# ============================================================

slope = ee.Terrain.slope(
    dem
)

slope_result = (
    slope
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=30,
        bestEffort=True
    )
)

slope_value = slope_result.get(
    "slope"
).getInfo()

print("\n📐 SLOPE")
print("Mean slope:", slope_value, "degrees")


# ============================================================
# 7. CREATE LIVE DATAFRAME
# ============================================================

live_data = pd.DataFrame([{
    "rainfall": rainfall_value,
    "soil_moisture": soil_value,
    "elevation": elevation,
    "slope": slope_value,
    "rainfall_datetime": rain_date_value,
    "soil_moisture_date": sm_date_value
}])

print("\n========================================")
print("           LIVE GEE DATA")
print("========================================")

display(live_data)

✅ Google Earth Engine initialized
✅ Joshimath ROI ready

🌧️ LIVE RAINFALL
Date: 2026-09-04 22:30
Rainfall: 0.5665837624266341

💧 LIVE SOIL MOISTURE
Date: 2023-12-02
Soil moisture: None

⛰️ DEM
Mean elevation: 2721.837517392416 m

📐 SLOPE
Mean slope: 35.28230268143529 degrees

           LIVE GEE DATA


,rainfall,soil_moisture,elevation,slope,rainfall_datetime,soil_moisture_date
0,0.566584,None,2721.837517,35.282303,2026-09-04 22:30,2023-12-02


In [2]:
import ee

ee.Initialize(project="gis-project-work")

roi = ee.Geometry.Point([
    79.56436,
    30.55543
]).buffer(10000)

smap = (
    ee.ImageCollection("NASA/SMAP/SPL3SMP_E/005")
    .filterBounds(roi)
    .sort("system:time_start", False)
)

print("Total SMAP images:")
print(smap.size().getInfo())

latest = ee.Image(smap.first())

print("\nLatest SMAP image:")
print(latest.getInfo()["id"])

print("\nAvailable bands:")
print(latest.bandNames().getInfo())

print("\nDate:")
print(
    ee.Date(
        latest.get("system:time_start")
    ).format("YYYY-MM-dd HH:mm").getInfo()
)

Total SMAP images:
3082

Latest SMAP image:
NASA/SMAP/SPL3SMP_E/005/20231202

Available bands:
['soil_moisture_am', 'tb_h_corrected_am', 'tb_v_corrected_am', 'vegetation_water_content_am', 'retrieval_qual_flag_am', 'tb_qual_flag_h_am', 'tb_qual_flag_v_am', 'soil_moisture_pm', 'tb_h_corrected_pm', 'tb_v_corrected_pm', 'vegetation_water_content_pm', 'retrieval_qual_flag_pm', 'tb_qual_flag_h_pm', 'tb_qual_flag_v_pm', 'soil_moisture_am_anomaly', 'soil_moisture_pm_anomaly']

Date:
2023-12-02 00:00


In [3]:
bands = latest.bandNames().getInfo()

print("Bands found:")
for band in bands:
    print("-", band)

Bands found:
- soil_moisture_am
- tb_h_corrected_am
- tb_v_corrected_am
- vegetation_water_content_am
- retrieval_qual_flag_am
- tb_qual_flag_h_am
- tb_qual_flag_v_am
- soil_moisture_pm
- tb_h_corrected_pm
- tb_v_corrected_pm
- vegetation_water_content_pm
- retrieval_qual_flag_pm
- tb_qual_flag_h_pm
- tb_qual_flag_v_pm
- soil_moisture_am_anomaly
- soil_moisture_pm_anomaly


In [4]:
import ee

ee.Initialize(project="gis-project-work")

# Joshimath 10 km ROI
roi = ee.Geometry.Point([
    79.56436,
    30.55543
]).buffer(10000)

# SMAP collection
smap = (
    ee.ImageCollection("NASA/SMAP/SPL3SMP_E/005")
    .filterBounds(roi)
    .sort("system:time_start", False)
)


# ============================================================
# CHECK RECENT SMAP IMAGES FOR VALID SOIL MOISTURE
# ============================================================

def check_image(image):

    value = (
        image
        .select("soil_moisture_am")
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=roi,
            scale=10000,
            bestEffort=True
        )
        .get("soil_moisture_am")
    )

    return image.set(
        "roi_soil_moisture",
        value
    )


checked = smap.map(check_image)

# Keep only images where soil moisture exists
valid = checked.filter(
    ee.Filter.notNull(["roi_soil_moisture"])
)

print("Total SMAP images:", smap.size().getInfo())
print("Images with valid Joshimath soil moisture:", valid.size().getInfo())


# ============================================================
# GET LATEST VALID IMAGE
# ============================================================

latest_valid = ee.Image(
    valid.sort("system:time_start", False).first()
)

date = ee.Date(
    latest_valid.get("system:time_start")
).format("YYYY-MM-dd HH:mm").getInfo()

soil_moisture = latest_valid.get(
    "roi_soil_moisture"
).getInfo()

print("\n===================================")
print("LATEST VALID SOIL MOISTURE")
print("===================================")

print("Date:", date)
print("Soil moisture:", soil_moisture)

Total SMAP images: 3082


EEException: User memory limit exceeded.

In [5]:
valid = checked.filter(
    ee.Filter.notNull(["roi_soil_moisture"])
)

In [6]:
import ee

ee.Initialize(project="gis-project-work")

# Joshimath
joshimath = ee.Geometry.Point([
    79.56436,
    30.55543
])

roi = joshimath.buffer(10000)


# ============================================================
# ONLY CHECK RECENT SMAP DATA
# ============================================================

smap = (
    ee.ImageCollection("NASA/SMAP/SPL3SMP_E/005")
    .filterBounds(roi)
    .filterDate(
        "2023-10-01",
        "2024-01-01"
    )
    .sort("system:time_start", False)
)

print(
    "SMAP images in recent period:",
    smap.size().getInfo()
)


# Get latest image from this small collection
latest = ee.Image(smap.first())

print(
    "Latest image:",
    latest.get("system:index").getInfo()
)

print(
    "Date:",
    ee.Date(
        latest.get("system:time_start")
    ).format(
        "YYYY-MM-dd"
    ).getInfo()
)


# ============================================================
# CHECK SOIL MOISTURE
# ============================================================

soil = (
    latest
    .select("soil_moisture_am")
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=9000,
        bestEffort=True,
        maxPixels=1e7
    )
)

print(
    "Soil moisture:",
    soil.get("soil_moisture_am").getInfo()
)

SMAP images in recent period: 62
Latest image: 20231202
Date: 2023-12-02
Soil moisture: None


In [7]:
import ee

ee.Initialize(project="gis-project-work")

# Joshimath point
point = ee.Geometry.Point([
    79.56436,
    30.55543
])

# SMAP
smap = (
    ee.ImageCollection("NASA/SMAP/SPL3SMP_E/005")
    .filterDate("2023-12-01", "2023-12-03")
)

print("Images:", smap.size().getInfo())

image = ee.Image(smap.first())

print("Image:", image.get("system:index").getInfo())


# Check AM soil moisture at exact Joshimath point
result = image.select(
    "soil_moisture_am"
).reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=9000,
    bestEffort=True
)

print(
    "Joshimath point soil moisture:",
    result.get("soil_moisture_am").getInfo()
)


# Check PM as well
result_pm = image.select(
    "soil_moisture_pm"
).reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=9000,
    bestEffort=True
)

print(
    "Joshimath point PM soil moisture:",
    result_pm.get("soil_moisture_pm").getInfo()
)

Images: 2
Image: 20231201
Joshimath point soil moisture: None
Joshimath point PM soil moisture: None


In [8]:
import ee

ee.Initialize(project="gis-project-work")

# ============================================================
# JOSHIMATH
# ============================================================

joshimath = ee.Geometry.Point([
    79.56436,
    30.55543
])

roi = joshimath.buffer(10000)


# ============================================================
# SMAP L4 — 3-HOURLY SOIL MOISTURE
# ============================================================

smap_l4 = (
    ee.ImageCollection("NASA/SMAP/SPL4SMGP/008")
    .filterBounds(roi)
    .sort("system:time_start", False)
)

print(
    "Total SMAP L4 images:",
    smap_l4.size().getInfo()
)


# ============================================================
# LATEST IMAGE
# ============================================================

latest = ee.Image(
    smap_l4.first()
)

date = ee.Date(
    latest.get("system:time_start")
).format(
    "YYYY-MM-dd HH:mm"
).getInfo()

print("Latest SMAP L4:", date)


# ============================================================
# SURFACE SOIL MOISTURE
# 0–5 cm
# ============================================================

surface_sm = (
    latest
    .select("sm_surface")
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=11000,
        bestEffort=True,
        maxPixels=1e7
    )
)

surface_value = surface_sm.get(
    "sm_surface"
).getInfo()


# ============================================================
# ROOT-ZONE SOIL MOISTURE
# 0–100 cm
# ============================================================

rootzone_sm = (
    latest
    .select("sm_rootzone")
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=11000,
        bestEffort=True,
        maxPixels=1e7
    )
)

rootzone_value = rootzone_sm.get(
    "sm_rootzone"
).getInfo()


# ============================================================
# RESULT
# ============================================================

print("\n========================================")
print("       LIVE SMAP L4 SOIL MOISTURE")
print("========================================")

print("Date:", date)
print("Surface soil moisture:", surface_value)
print("Root-zone soil moisture:", rootzone_value)

Total SMAP L4 images: 33382
Latest SMAP L4: 2026-09-02 22:30

       LIVE SMAP L4 SOIL MOISTURE
Date: 2026-09-02 22:30
Surface soil moisture: 0.32442243893941247
Root-zone soil moisture: 0.324946383635203
